# FPGA Star Tracker — PYNQ Z2 Notebook

This notebook is the FPGA-side counterpart to the `Baseline_CPU_version` Python scripts.  
It runs the **same geometric feature extraction and classifier pipeline** so results are directly comparable.

**Pipeline overview:**
1. Generate a synthetic starfield dataset (same rules as `star_data.py`)
2. Extract 12-element hand-crafted feature vectors (same as `train_star.py`)
3. Train a Random Forest classifier
4. Benchmark inference latency and accuracy
5. Live inference from USB camera

**Target board:** PYNQ Z2 (Zynq-7020, ARM Cortex-A9, 512 MB DDR3)

## 0. Install / verify dependencies
Run this cell once after a fresh PYNQ image to ensure all packages are present.

In [ ]:
import subprocess, sys

required = ["numpy", "Pillow", "scikit-learn", "joblib", "matplotlib", "seaborn", "opencv-python-headless"]
for pkg in required:
    try:
        __import__(pkg.split("-")[0].replace("-", "_").lower())
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All dependencies satisfied.")

## 1. Imports and Configuration

In [ ]:
import os
import random
import time
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import cv2

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ── Image / dataset config (mirrors star_data.py) ──────────────────────────
IMG_W            = 320
IMG_H            = 240
IMAGES_PER_CLASS = 500       # reduce to 200 if storage is tight on PYNQ
DATASET_DIR      = "starfield_dataset"
CLASSES          = ["up", "down", "left", "right", "forward", "back"]
NUM_CLASSES      = len(CLASSES)
CLASS_TO_ID      = {c: i for i, c in enumerate(CLASSES)}
ID_TO_CLASS      = {i: c for c, i in CLASS_TO_ID.items()}

MIN_STARS            = 40
MAX_STARS            = 120
STAR_RADIUS          = 1
STAR_BRIGHTNESS_MIN  = 180
STAR_BRIGHTNESS_MAX  = 255
BIAS_RATIO           = 0.70
FORWARD_MIN_RADIAL   = 0.55
FORWARD_MAX_RADIAL   = 0.90
BACK_MIN_RADIAL      = 0.05
BACK_MAX_RADIAL      = 0.40

# ── Classifier config (mirrors train_star.py) ──────────────────────────────
STAR_THRESHOLD = 100          # brightness threshold for star detection
CLASSIFIER     = "rf"         # "logreg" | "rf" | "gbm"

FEATURE_NAMES = [
    "top_ratio", "bottom_ratio", "left_ratio", "right_ratio",
    "centroid_x", "centroid_y", "mean_radial",
    "star_density", "center_density", "edge_density",
    "std_x", "std_y",
]

print(f"Config loaded. IMG={IMG_W}x{IMG_H}, classes={CLASSES}, classifier={CLASSIFIER.upper()}")

## 2. Camera Functions
Adapted from `FPGA_version/camera_driver.ipynb`. The camera is used in Section 7 for live inference.

In [ ]:
def camera_init():
    """Open USB camera via V4L2. Returns the VideoCapture object."""
    camera = cv2.VideoCapture(0, cv2.CAP_V4L2)
    camera.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'MJPG'))
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    # Warm-up: flush stale frames
    for _ in range(15):
        ret, frame = camera.read()
        if frame is not None and frame.max() > 0:
            break
        time.sleep(0.1)
    print("Camera ready:", ret and frame.max() > 0)
    return camera


def get_cam_image(camera, display=True):
    """Flush stale frames and capture a fresh BGR frame."""
    for _ in range(10):
        camera.grab()
    ret, frame = camera.read()
    if not ret or frame.max() <= 2:
        raise RuntimeError("Failed to capture a valid frame.")
    if display:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(6, 4))
        plt.imshow(frame_rgb)
        plt.title('USB Camera Capture')
        plt.axis('off')
        plt.show()
    return frame


def deinit_camera(camera):
    """Release the camera resource."""
    camera.release()
    time.sleep(2)
    print("Camera released.")


def frame_to_grayscale_array(frame):
    """
    Convert a BGR camera frame to a grayscale numpy array
    resized to match the training resolution (IMG_W x IMG_H).
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, (IMG_W, IMG_H))
    return resized.astype(np.uint8)


print("Camera functions defined.")

## 3. Synthetic Dataset Generation
Identical logic to `star_data.py`. Generates 6-class starfield images.

In [ ]:
# ── Placement helpers ──────────────────────────────────────────────────────

def random_star(x_range, y_range):
    return random.uniform(*x_range), random.uniform(*y_range)

def random_star_full():
    return random_star((0, IMG_W), (0, IMG_H))

def radial_star(min_r_frac, max_r_frac):
    cx, cy = IMG_W / 2, IMG_H / 2
    half_diag = np.sqrt(cx**2 + cy**2)
    min_r, max_r = min_r_frac * half_diag, max_r_frac * half_diag
    for _ in range(200):
        r = random.uniform(min_r, max_r)
        theta = random.uniform(0, 2 * np.pi)
        x, y = cx + r * np.cos(theta), cy + r * np.sin(theta)
        if 0 <= x < IMG_W and 0 <= y < IMG_H:
            return x, y
    return random_star_full()

def place_stars(n_stars, biased_fn, noise_fn=random_star_full):
    n_biased = int(round(n_stars * BIAS_RATIO))
    stars = [biased_fn() for _ in range(n_biased)]
    stars += [noise_fn() for _ in range(n_stars - n_biased)]
    random.shuffle(stars)
    return stars

# ── Per-class generators ────────────────────────────────────────────────────

def stars_up(n):      return place_stars(n, lambda: random_star((0, IMG_W),       (0,           IMG_H * 0.5)))
def stars_down(n):    return place_stars(n, lambda: random_star((0, IMG_W),       (IMG_H * 0.5, IMG_H)))
def stars_left(n):    return place_stars(n, lambda: random_star((0, IMG_W * 0.5), (0,           IMG_H)))
def stars_right(n):   return place_stars(n, lambda: random_star((IMG_W * 0.5, IMG_W), (0,       IMG_H)))
def stars_forward(n): return place_stars(n, lambda: radial_star(FORWARD_MIN_RADIAL, FORWARD_MAX_RADIAL))
def stars_back(n):    return place_stars(n, lambda: radial_star(BACK_MIN_RADIAL,    BACK_MAX_RADIAL))

CLASS_GENERATORS = {
    "up": stars_up, "down": stars_down, "left": stars_left,
    "right": stars_right, "forward": stars_forward, "back": stars_back,
}

# ── Renderer ────────────────────────────────────────────────────────────────

def render_starfield(star_positions):
    img  = Image.new("L", (IMG_W, IMG_H), color=0)
    draw = ImageDraw.Draw(img)
    for (x, y) in star_positions:
        brightness = random.randint(STAR_BRIGHTNESS_MIN, STAR_BRIGHTNESS_MAX)
        xi, yi = int(x), int(y)
        if STAR_RADIUS == 0:
            if 0 <= xi < IMG_W and 0 <= yi < IMG_H:
                img.putpixel((xi, yi), brightness)
        else:
            r = STAR_RADIUS
            draw.ellipse([(xi - r, yi - r), (xi + r, yi + r)], fill=brightness)
    return img

# ── Dataset generation ──────────────────────────────────────────────────────

def generate_class(class_name, out_dir, n_images):
    os.makedirs(out_dir, exist_ok=True)
    generator = CLASS_GENERATORS[class_name]
    for i in range(n_images):
        n_stars = random.randint(MIN_STARS, MAX_STARS)
        img = render_starfield(generator(n_stars))
        img.save(os.path.join(out_dir, f"{i:04d}.png"))
    return n_images


def generate_all(skip_existing=True):
    print("=" * 50)
    print("  STAR TRACKER DATASET GENERATOR")
    print(f"  Resolution : {IMG_W} x {IMG_H}")
    print(f"  Images/cls : {IMAGES_PER_CLASS}")
    print(f"  Bias ratio : {BIAS_RATIO:.0%}")
    print("=" * 50)
    total = 0
    for class_name in CLASSES:
        out_dir = os.path.join(DATASET_DIR, class_name)
        existing = len([f for f in os.listdir(out_dir) if f.endswith(".png")]) if os.path.exists(out_dir) else 0
        if skip_existing and existing >= IMAGES_PER_CLASS:
            print(f"  {class_name:>8s}: skipped ({existing} images already exist)")
            total += existing
            continue
        print(f"  {class_name:>8s}: generating...", end=" ", flush=True)
        n = generate_class(class_name, out_dir, IMAGES_PER_CLASS)
        print(f"{n} images saved.")
        total += n
    print(f"\n  Done. Total images: {total}")
    print("=" * 50)


print("Dataset generation functions defined.")

### 3a. Run Dataset Generation
Set `skip_existing=True` to avoid regenerating images that already exist on disk.

In [ ]:
generate_all(skip_existing=True)

### 3b. Preview Generated Images

In [ ]:
fig, axes = plt.subplots(1, len(CLASSES), figsize=(14, 3))
for ax, class_name in zip(axes, CLASSES):
    sample_path = os.path.join(DATASET_DIR, class_name, "0000.png")
    img = Image.open(sample_path)
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(class_name)
    ax.axis("off")
plt.suptitle("One sample per class", y=1.02)
plt.tight_layout()
plt.show()

## 4. Feature Extraction
12-element hand-crafted geometric feature vector — identical to `train_star.py`.

| Index | Name | Description |
|-------|------|-------------|
| 0 | top_ratio | fraction of stars in top half |
| 1 | bottom_ratio | fraction of stars in bottom half |
| 2 | left_ratio | fraction of stars in left half |
| 3 | right_ratio | fraction of stars in right half |
| 4 | centroid_x | normalized horizontal centroid |
| 5 | centroid_y | normalized vertical centroid |
| 6 | mean_radial | mean radial distance / half-diagonal |
| 7 | star_density | normalized star count |
| 8 | center_density | fraction within inner 25% of half-diag |
| 9 | edge_density | fraction beyond 75% of half-diag |
| 10 | std_x | normalized horizontal spread |
| 11 | std_y | normalized vertical spread |

In [ ]:
def detect_stars(img_array):
    """
    Return (N, 2) float32 array of (x, y) coordinates where
    pixel brightness > STAR_THRESHOLD.
    Same logic as brightness comparator in FPGA HDL.
    """
    ys, xs = np.where(img_array > STAR_THRESHOLD)
    if len(xs) == 0:
        return np.empty((0, 2), dtype=np.float32)
    return np.column_stack([xs, ys]).astype(np.float32)


def extract_features(img_array):
    """
    Compute the 12-element feature vector for one grayscale image (H, W).
    Returns np.ndarray shape (12,), dtype float32.
    """
    stars = detect_stars(img_array)
    n = len(stars)
    if n == 0:
        return np.zeros(12, dtype=np.float32)

    xs, ys = stars[:, 0], stars[:, 1]
    cx_img, cy_img = IMG_W / 2.0, IMG_H / 2.0
    half_diag = np.sqrt(cx_img**2 + cy_img**2)

    top_ratio    = np.sum(ys < cy_img) / n
    bottom_ratio = 1.0 - top_ratio
    left_ratio   = np.sum(xs < cx_img) / n
    right_ratio  = 1.0 - left_ratio

    centroid_x = np.mean(xs) / IMG_W
    centroid_y = np.mean(ys) / IMG_H

    dx = xs - cx_img
    dy = ys - cy_img
    radial_dists = np.sqrt(dx**2 + dy**2)
    mean_radial  = np.mean(radial_dists) / half_diag

    center_density = np.sum(radial_dists < 0.25 * half_diag) / n
    edge_density   = np.sum(radial_dists > 0.75 * half_diag) / n
    star_density   = min(n / MAX_STARS, 1.0)

    std_x = np.std(xs) / IMG_W
    std_y = np.std(ys) / IMG_H

    return np.array([
        top_ratio, bottom_ratio, left_ratio, right_ratio,
        centroid_x, centroid_y, mean_radial,
        star_density, center_density, edge_density,
        std_x, std_y,
    ], dtype=np.float32)


print("Feature extraction functions defined.")

## 5. Load Dataset and Extract Features

In [ ]:
def load_and_extract(dataset_dir):
    """
    Load all PNG images, run feature extraction.
    Returns X (N, 12) float32 and y (N,) int32.
    """
    X, y = [], []
    print("Loading images and extracting features...")
    for class_name in CLASSES:
        class_dir = os.path.join(dataset_dir, class_name)
        if not os.path.exists(class_dir):
            print(f"  WARNING: folder not found: {class_dir}")
            continue
        files = [f for f in os.listdir(class_dir) if f.endswith(".png")]
        print(f"  {class_name:>8s}: {len(files)} images", end=" ", flush=True)
        class_features = []
        for fname in files:
            img = Image.open(os.path.join(class_dir, fname)).convert("L").resize((IMG_W, IMG_H))
            arr = np.array(img, dtype=np.uint8)
            feat = extract_features(arr)
            class_features.append(feat)
            X.append(feat)
            y.append(CLASS_TO_ID[class_name])
        cf = np.array(class_features)
        print(f"| top={cf[:,0].mean():.2f} bot={cf[:,1].mean():.2f} "
              f"lft={cf[:,2].mean():.2f} rgt={cf[:,3].mean():.2f} "
              f"rad={cf[:,6].mean():.2f}")
    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int32)
    print(f"\nTotal: {len(X)} samples | Feature dim: {X.shape[1]}")
    return X, y


X, y = load_and_extract(DATASET_DIR)

## 6. Train / Val / Test Split and Scaling

In [ ]:
# 70 / 15 / 15 split — mirrors train_star.py
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Split — Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print("Features scaled.")

## 7. Build and Train Classifier

In [ ]:
def build_classifier(name):
    if name == "logreg":
        return LogisticRegression(max_iter=1000, C=1.0, multi_class="multinomial", solver="lbfgs")
    elif name == "rf":
        return RandomForestClassifier(n_estimators=200, max_depth=None,
                                      min_samples_leaf=2, random_state=42, n_jobs=-1)
    elif name == "gbm":
        return GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                          learning_rate=0.1, random_state=42)
    raise ValueError(f"Unknown classifier: {name}")


print(f"Training {CLASSIFIER.upper()} classifier...")
clf = build_classifier(CLASSIFIER)
t0 = time.perf_counter()
clf.fit(X_train_s, y_train)
train_time = time.perf_counter() - t0
print(f"Training complete in {train_time:.2f}s")

val_acc = accuracy_score(y_val, clf.predict(X_val_s))
print(f"Validation accuracy: {val_acc * 100:.2f}%")

## 8. Cross-Validation

In [ ]:
print("Running 5-fold cross-validation...")
cv_scores = cross_val_score(clf, X_train_s, y_train, cv=5, n_jobs=-1)
print(f"CV scores: {cv_scores * 100}")
print(f"Mean: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

## 9. Test Set Evaluation

In [ ]:
y_pred   = clf.predict(X_test_s)
test_acc = accuracy_score(y_test, y_pred)
report_str  = classification_report(y_test, y_pred, target_names=CLASSES)
report_dict = classification_report(y_test, y_pred, target_names=CLASSES, output_dict=True)

print(f"Test accuracy: {test_acc * 100:.2f}%\n")
print(report_str)

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("Confusion Matrix — Geometric Feature Classifier (PYNQ)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix_pynq.png", dpi=150)
print("Saved: confusion_matrix_pynq.png")
plt.show()

### Per-Class F1 Score

In [ ]:
classes  = [c for c in CLASSES if c in report_dict]
f1_scores = [report_dict[c]["f1-score"] for c in classes]

plt.figure(figsize=(8, 4))
bars = plt.bar(classes, f1_scores, color="steelblue", edgecolor="white")
plt.ylim(0, 1.05)
plt.axhline(0.9, color="red", linestyle="--", linewidth=1, label="90% target")
for bar, val in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f"{val:.2f}", ha="center", va="bottom", fontsize=9)
plt.title("Per-Class F1 Score (PYNQ)")
plt.ylabel("F1 Score")
plt.legend()
plt.tight_layout()
plt.savefig("training_curves_pynq.png", dpi=150)
print("Saved: training_curves_pynq.png")
plt.show()

### Feature Importance (Random Forest)

In [ ]:
if hasattr(clf, "feature_importances_"):
    importances = clf.feature_importances_
    idx = np.argsort(importances)[::-1]
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(importances)), importances[idx])
    plt.xticks(range(len(importances)), [FEATURE_NAMES[i] for i in idx], rotation=45, ha="right")
    plt.title("Feature Importances (Random Forest — PYNQ)")
    plt.tight_layout()
    plt.savefig("feature_importance_pynq.png", dpi=150)
    print("Saved: feature_importance_pynq.png")
    plt.show()
else:
    print("Feature importance not available for this classifier.")

## 10. Benchmark Inference Time
Measures end-to-end latency (feature extract → scale → predict) over 1000 runs,
mirroring the benchmark in `train_star.py` for direct comparison.

In [ ]:
def benchmark_inference(clf, scaler, num_runs=1000):
    """
    Time: feature_extract + scale + predict for a single image.
    Returns (avg_latency_ms, throughput_img_per_sec).
    """
    sample_dir  = os.path.join(DATASET_DIR, CLASSES[0])
    sample_file = [f for f in os.listdir(sample_dir) if f.endswith(".png")][0]
    sample_img  = np.array(
        Image.open(os.path.join(sample_dir, sample_file)).convert("L").resize((IMG_W, IMG_H)),
        dtype=np.uint8
    )

    # Warm up
    feat = extract_features(sample_img).reshape(1, -1)
    _ = clf.predict(scaler.transform(feat))

    # Timed runs
    start = time.perf_counter()
    for _ in range(num_runs):
        feat = extract_features(sample_img).reshape(1, -1)
        _ = clf.predict(scaler.transform(feat))
    elapsed = time.perf_counter() - start

    avg_ms     = (elapsed / num_runs) * 1000
    throughput = 1000.0 / avg_ms
    return avg_ms, throughput


print("Benchmarking inference time (1000 runs)...")
avg_ms, throughput = benchmark_inference(clf, scaler)
print(f"Avg latency : {avg_ms:.4f} ms")
print(f"Throughput  : {throughput:.0f} images / sec")

## 11. Save Model and Benchmark Report

In [ ]:
joblib.dump(clf,    "star_tracker_model_pynq.pkl")
joblib.dump(scaler, "feature_scaler_pynq.pkl")
print("Saved: star_tracker_model_pynq.pkl")
print("Saved: feature_scaler_pynq.pkl")

report_lines = [
    "=" * 55,
    "  FPGA STAR TRACKER — PYNQ Z2 BENCHMARK REPORT",
    "=" * 55,
    "",
    f"  Platform           : PYNQ Z2 (Zynq-7020, ARM Cortex-A9)",
    f"  Classifier         : {CLASSIFIER.upper()}",
    f"  Feature Dim        : 12 (hand-crafted spatial features)",
    f"  Image Resolution   : {IMG_W} x {IMG_H} px (grayscale)",
    f"  Classes            : {', '.join(CLASSES)}",
    "",
    "  ── Accuracy ──",
    f"  Test Accuracy      : {test_acc * 100:.2f}%",
    f"  Cross-Val (5-fold) : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%",
    "",
    "  ── Inference Timing (PYNQ ARM, per image) ──",
    f"  Avg Latency        : {avg_ms:.4f} ms",
    f"  Throughput         : {throughput:.0f} images / sec",
    "",
    "  ── Per-Class Report ──",
    report_str,
    "",
    "  NOTE: Feature extraction logic is identical to the",
    "  Baseline_CPU_version for direct comparison.",
    "=" * 55,
]
report_text = "\n".join(report_lines)
print(report_text)

with open("benchmark_report_pynq.txt", "w") as f:
    f.write(report_text)
print("\nSaved: benchmark_report_pynq.txt")

## 12. Live Inference from USB Camera

Connects the trained model to the USB camera.  
Each captured frame is converted to grayscale, resized to 320×240, and classified using the same feature pipeline.

> **Note:** If `camera_init()` fails on first run, re-execute that cell — the PYNQ USB driver sometimes needs a second attempt.

In [ ]:
# Initialize camera — re-run this cell if "Camera ready: False"
camera = camera_init()

In [ ]:
def classify_frame(camera, clf, scaler, display=True):
    """
    Capture one frame, extract features, and return the predicted class.
    Optionally displays the frame with the prediction overlaid.
    """
    frame    = get_cam_image(camera, display=False)
    gray_arr = frame_to_grayscale_array(frame)

    feat        = extract_features(gray_arr).reshape(1, -1)
    feat_scaled = scaler.transform(feat)
    pred_id     = clf.predict(feat_scaled)[0]
    pred_class  = ID_TO_CLASS[pred_id]
    proba       = clf.predict_proba(feat_scaled)[0] if hasattr(clf, "predict_proba") else None

    if display:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        fig, axes = plt.subplots(1, 2, figsize=(10, 4),
                                 gridspec_kw={"width_ratios": [2, 1]})

        axes[0].imshow(frame_rgb)
        axes[0].set_title(f"Prediction: {pred_class.upper()}", fontsize=14, fontweight="bold")
        axes[0].axis("off")

        if proba is not None:
            colors = ["steelblue"] * NUM_CLASSES
            colors[pred_id] = "tomato"
            axes[1].barh(CLASSES, proba, color=colors)
            axes[1].set_xlim(0, 1)
            axes[1].set_xlabel("Probability")
            axes[1].set_title("Class probabilities")

        plt.tight_layout()
        plt.show()

    return pred_class, proba


# Single classification
pred_class, proba = classify_frame(camera, clf, scaler, display=True)
print(f"Predicted orientation: {pred_class}")

In [ ]:
# Continuous classification — run N frames and print results
N_FRAMES = 5

latencies = []
for i in range(N_FRAMES):
    t0 = time.perf_counter()
    pred, _ = classify_frame(camera, clf, scaler, display=True)
    elapsed_ms = (time.perf_counter() - t0) * 1000
    latencies.append(elapsed_ms)
    print(f"Frame {i+1}/{N_FRAMES} → {pred:>8s}  ({elapsed_ms:.1f} ms end-to-end)")

print(f"\nAvg end-to-end latency (incl. capture): {np.mean(latencies):.1f} ms")

In [ ]:
# Release camera when done
deinit_camera(camera)